In [ ]:
from pathlib import Path
import sys,json,re,html,difflib
from IPython.display import display
P=Path('/home/ec2-user/SageMaker/election_arm')
B=P/'source_review_main_ai_v1'
sys.path.insert(0,str(P/'source_review_pilot_tdm_v1/widget_vendor'))
import ipywidgets as w
sources=json.loads((B/'sources.json').read_text())
pack=json.loads((B/'pack.json').read_text())
labels={r['audit_id']:r for r in json.loads((B/'comparison.json').read_text())['records']}
state={'i':0,'labels':False,'offset':0}
header=w.HTML()
body=w.HTML()
prev=w.Button(description='Previous 2')
nxt=w.Button(description='Next 2')
reveal=w.Button(description='Show labels')
more=w.Button(description='More context')
reset=w.Button(description='Reset context')
def clean(t): return re.sub(r'\s+',' ',t).strip()
def card(i):
 r,s=pack['records'][i],sources[i]
 q,t=clean(s.get('quote','')),clean(s.get('article',''))
 pos=t.find(q)
 method='exact/whitespace' if pos>=0 else 'approximate locator: inspect evidence'
 if pos<0:
  match=difflib.SequenceMatcher(None,q.lower(),t.lower(),autojunk=False).find_longest_match()
  pos=max(0,match.b-match.a)
 start=max(0,pos-200)+state['offset']
 context=t[start:start+1100]
 text=f"<h3>{i+1}. {r['date']} | {r['audit_id']}</h3><p>{html.escape(r['publisher'])} | raw exact: {s.get('quote_occurrences')} | {method}</p><b>CLAIM</b><p>{html.escape(q)}</p><b>ARTICLE CONTEXT ({start}:{start+len(context)} of {len(t)})</b><p>{html.escape(context)}</p>"
 if state['labels']:
  v=labels[r['audit_id']]
  text+='<hr><b>MACHINE LABELS</b><p>'+html.escape(' | '.join(k+'='+str(v[k]) for k in v if k!='audit_id'))+'</p>'
 return '<div style="width:49%;padding:0 12px;box-sizing:border-box;font:14px/1.35 Arial">'+text+'</div>'
def render():
 header.value=f"<b>AI source review {state['i']+1}-{min(state['i']+2,100)} / 100</b> | Recovered {sum(s['recovered'] for s in sources)}/100 | Exact quotes {sum(s.get('quote_occurrences')==1 for s in sources)}/100 | No human responses written"
 body.value='<div style="display:flex;align-items:flex-start">'+''.join(card(i) for i in range(state['i'],min(state['i']+2,100)))+'</div>'
def move(delta):
 state.update(i=max(0,min(98,state['i']+delta)),labels=False,offset=0);render()
prev.on_click(lambda _:move(-2));nxt.on_click(lambda _:move(2))
def reveal_now(_): state['labels']=True;render()
def more_now(_): state['offset']+=900;render()
def reset_now(_): state['offset']=0;render()
reveal.on_click(reveal_now);more.on_click(more_now);reset.on_click(reset_now)
render()
display(w.VBox([header,w.HBox([prev,nxt,reveal,more,reset]),body]))
